# AI vs Real Face Detector — Full Hybrid Colab Training

## Run 2: FFHQ + StyleGAN2 + Diffusion

This notebook trains the new `full_hybrid` model:

```text
Face
 ├── EfficientNet
 ├── Physics + Fresnel + Shadow
 ├── PRNU
 └── ViT / Semantic
          ↓
     Gated Fusion
          ↓
       Classifier
          ↓
 REAL / AI / UNCERTAIN
```

**GPU:** T4 recommended.

The old `stage1` and `hybrid` checkpoints remain available as baselines, but this notebook's main experiment is `full_hybrid`.

## 1. GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), "Enable Runtime → Change runtime type → T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

## 2. Clone the UPDATED repository

In [ ]:
%cd /content
!rm -rf /content/ai-vs-real-face-detector

# IMPORTANT: push the Cursor changes to GitHub before running this cell.
!git clone --branch master --single-branch https://github.com/Algorithm-bot/ai-vs-real-face-detector.git /content/ai-vs-real-face-detector

REPO = "/content/ai-vs-real-face-detector"
PROJECT = f"{REPO}/ai-vs-real-face-detector"

!grep -n "full_hybrid" "{PROJECT}/src/train.py" | head -20

## 3. Install project dependencies

In [ ]:
%cd /content/ai-vs-real-face-detector/ai-vs-real-face-detector
!pip -q install -r requirements.txt

## 4. Mount Drive and verify Run 2 dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

DATA_DIR = Path('/content/drive/MyDrive/ai-vs-real-face-detector/data')
OUTPUT_DIR = Path('/content/drive/MyDrive/ai-vs-real-face-detector/models')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for split in ['train', 'val', 'test']:
    for label in ['real', 'fake']:
        path = DATA_DIR / split / label
        assert path.is_dir(), f"Missing: {path}"

        count = sum(
            1 for p in path.rglob('*')
            if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}
        )
        print(f"{split}/{label}: {count}")

print("\nData:", DATA_DIR)
print("Models:", OUTPUT_DIR)

## 5. Full-hybrid smoke test

Run this first. It checks the complete multimodal path before spending GPU time on a 15-epoch run.

In [ ]:
!python "{PROJECT}/src/train.py"     --mode full_hybrid     --data-dir "{DATA_DIR}"     --output-dir "{OUTPUT_DIR}/smoke_test"     --epochs 1     --batch-size 2     --num-workers 0     --fusion-mode gated

## 6. Run 2 — Full Hybrid Training

**Start with batch size 16 on a T4.**

The model uses real:
- EfficientNet features
- physics features
- PRNU features
- ViT/semantic features

No placeholder vectors.

In [ ]:
!python "{PROJECT}/src/train.py"     --mode full_hybrid     --data-dir "{DATA_DIR}"     --output-dir "{OUTPUT_DIR}"     --epochs 15     --batch-size 16     --num-workers 2     --fusion-mode gated

## 7. Check the trained checkpoint

In [ ]:
import os, json

checkpoint = OUTPUT_DIR / "full_hybrid_best.pt"
history = OUTPUT_DIR / "full_hybrid_history.json"

print("Checkpoint exists:", checkpoint.exists(), checkpoint)
print("History exists:", history.exists(), history)

assert checkpoint.exists(), "full_hybrid_best.pt was not created."

if history.exists():
    with open(history) as f:
        h = json.load(f)
    print(json.dumps(h, indent=2)[:5000])

## 8. Final held-out test evaluation

The training script already records test metrics. This cell checks that the history contains them.

In [ ]:
if history.exists():
    with open(history) as f:
        h = json.load(f)

    print("Available history keys:", list(h.keys()))

    for key in ["test_accuracy", "test_precision", "test_recall", "test_f1", "test_roc_auc"]:
        if key in h:
            print(f"{key}: {h[key]}")

## 9. Optional: attention-fusion comparison

Do this **after** the gated run succeeds. It gives you an ablation between gated and attention fusion.

In [ ]:
# Uncomment only after the gated model has completed successfully.

# !python "{PROJECT}/src/train.py" #     --mode full_hybrid #     --data-dir "{DATA_DIR}" #     --output-dir "{OUTPUT_DIR}/attention_fusion" #     --epochs 15 #     --batch-size 16 #     --num-workers 2 #     --fusion-mode attention

## 10. Download the full-hybrid checkpoint

The checkpoint is already safely stored on Google Drive. Download it only if you need a local copy.

In [ ]:
from google.colab import files
files.download(str(OUTPUT_DIR / "full_hybrid_best.pt"))

# Run 2 complete

You now have a model trained on:

**Real:** FFHQ

**Fake:** StyleGAN2 + diffusion-generated faces

Next experiments:
1. Compare `full_hybrid_best.pt` against the old `hybrid_best.pt`.
2. Run the ablation study.
3. Test on an unseen generator if possible.
4. Calibrate the final model on the validation set before reporting final test confidence.